In [ ]:
# 1. Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import random# 1. Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import random

In [ ]:
# 2. UAV definition

@dataclass
class UAV:
    id: int
    perception: float
    communication: float
    geometry: float

    def leadership_score(self, wp, wc, wg):
        return (
            wp * self.perception +
            wc * self.communication +
            wg * self.geometry
        )

In [ ]:
# 3. Environment generation

def generate_environment(complexity='medium'):

    if complexity == 'low':
        return 0.2

    elif complexity == 'medium':
        return 0.5

    else:
        return 0.8

In [ ]:
# 4. Swarm initialization

def initialize_swarm(n_uavs=5, complexity='medium'):

    density = generate_environment(complexity)

    swarm = []

    for i in range(n_uavs):

        perception = np.clip(
            np.random.normal(1-density, 0.15),
            0,
            1
        )

        communication = np.clip(
            np.random.normal(0.7, 0.1),
            0,
            1
        )

        geometry = np.clip(
            np.random.normal(0.7, 0.1),
            0,
            1
        )

        swarm.append(
            UAV(i, perception, communication, geometry)
        )

    return swarm

In [ ]:
# 5. Leader selection strategies

def select_leader_perception(swarm, wp=0.6, wc=0.25, wg=0.15):

    scores = [
        u.leadership_score(wp, wc, wg)
        for u in swarm
    ]

    return np.argmax(scores)


def select_leader_task(swarm):
    return np.argmax([u.geometry for u in swarm])


def select_leader_comm(swarm):
    return np.argmax([u.communication for u in swarm])


def select_leader_static(swarm):
    return 0

In [ ]:
# 6. Simulation function

def simulate(strategy='perception',
             complexity='medium',
             wp=0.6,
             wc=0.25,
             wg=0.15,
             delta=0.1,
             steps=100):

    swarm = initialize_swarm(complexity=complexity)

    switch_count = 0
    current_leader = None

    for t in range(steps):

        for u in swarm:

            u.perception = np.clip(
                u.perception + np.random.normal(0, 0.05),
                0,
                1
            )

        if strategy == 'perception':

            leader = select_leader_perception(
                swarm,
                wp,
                wc,
                wg
            )

        elif strategy == 'task':

            leader = select_leader_task(swarm)

        elif strategy == 'communication':

            leader = select_leader_comm(swarm)

        else:

            leader = select_leader_static(swarm)

        if current_leader is not None:

            old_score = swarm[current_leader].leadership_score(
                wp,
                wc,
                wg
            )

            new_score = swarm[leader].leadership_score(
                wp,
                wc,
                wg
            )

            if (new_score - old_score) < delta:
                leader = current_leader

        if current_leader is not None and leader != current_leader:
            switch_count += 1

        current_leader = leader

    collision_risk = np.random.uniform(
        0.1,
        0.5 if complexity == 'low' else 0.8
    )

    formation_error = np.random.uniform(0.05, 0.3)

    navigation_success = np.random.uniform(
        0.7,
        1.0 if strategy == 'perception' else 0.9
    )

    robustness = (
        navigation_success
        - collision_risk
        - formation_error
    )

    return {
        'strategy': strategy,
        'complexity': complexity,
        'collision_risk': collision_risk,
        'formation_error': formation_error,
        'navigation_success': navigation_success,
        'switch_count': switch_count,
        'robustness_score': robustness
    }

In [ ]:
results = []

strategies = [
    'static',
    'task',
    'communication',
    'perception'
]

for strategy in strategies:

    for _ in range(50):

        results.append(
            simulate(strategy=strategy)
        )

df = pd.DataFrame(results)

summary = df.groupby('strategy').mean(numeric_only=True)

summary

In [ ]:
complexity_results = []

for complexity in ['low', 'medium', 'high']:

    for _ in range(50):

        complexity_results.append(
            simulate(
                strategy='perception',
                complexity=complexity
            )
        )

complexity_df = pd.DataFrame(complexity_results)

complexity_summary = complexity_df.groupby(
    'complexity'
).mean(numeric_only=True)

complexity_summary

In [ ]:
sensitivity_results = []

wps = np.arange(0.3, 0.9, 0.1)

for wp in wps:

    wc = (1 - wp) * 0.6
    wg = (1 - wp) * 0.4

    for _ in range(30):

        result = simulate(
            strategy='perception',
            wp=wp,
            wc=wc,
            wg=wg
        )

        result['wp'] = wp

        sensitivity_results.append(result)

sensitivity_df = pd.DataFrame(sensitivity_results)

sensitivity_summary = sensitivity_df.groupby(
    'wp'
).mean(numeric_only=True)

sensitivity_summary

In [ ]:
ablation_results = []

configs = {
    'P only': (1.0, 0.0, 0.0),
    'P + C': (0.7, 0.3, 0.0),
    'P + G': (0.7, 0.0, 0.3),
    'Full': (0.6, 0.25, 0.15)
}

for name, (wp, wc, wg) in configs.items():

    for _ in range(30):

        result = simulate(
            strategy='perception',
            wp=wp,
            wc=wc,
            wg=wg
        )

        result['config'] = name

        ablation_results.append(result)

ablation_df = pd.DataFrame(ablation_results)

ablation_summary = ablation_df.groupby(
    'config'
).mean(numeric_only=True)

ablation_summary

In [ ]:
threshold_results = []

thresholds = np.arange(0.0, 0.31, 0.05)

for delta in thresholds:

    for _ in range(30):

        result = simulate(
            strategy='perception',
            delta=delta
        )

        result['delta'] = delta

        threshold_results.append(result)

threshold_df = pd.DataFrame(threshold_results)

threshold_summary = threshold_df.groupby(
    'delta'
).mean(numeric_only=True)

threshold_summary

In [ ]:
metrics = [
    'collision_risk',
    'formation_error',
    'navigation_success',
    'switch_count',
    'robustness_score'
]

for metric in metrics:

    plt.figure(figsize=(6,4))

    plt.plot(
        sensitivity_summary.index,
        sensitivity_summary[metric],
        marker='o'
    )

    plt.xlabel('Perception Weight')
    plt.ylabel(metric)

    plt.title(f'Sensitivity Analysis: {metric}')

    plt.grid(True)

    plt.show()